# Single-GPU two-batch smoke test

This notebook runs exactly two real-data training steps in separate FP32 and AMP processes. It verifies finite loss and unscaled gradients, frozen backbone behavior, optimizer updates, peak GPU memory, and a full checkpoint round-trip. It does not run an epoch or start formal training.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


def required_path(name, directory=False):
    value = os.environ.get(name)
    assert value, f"Missing required environment variable: {name}"
    path = Path(value).expanduser().resolve()
    assert path.is_dir() if directory else path.exists(), f"{name} does not exist"
    return path


repo_root = required_path("DFDHR_REPO_ROOT", directory=True)
data_root = required_path("DFDHR_DATA_ROOT", directory=True)
runtime_root = required_path("DFDHR_RUNTIME_ROOT", directory=True)
checkpoint_path = required_path("DFDHR_OFFICIAL_CHECKPOINT")
json_root = Path(os.environ.get(
    "DFDHR_JSON_ROOT", repo_root / "preprocessing/dataset_json"
)).expanduser().resolve()
assert json_root.is_dir()
assert runtime_root != repo_root and repo_root not in runtime_root.parents
assert runtime_root != data_root and data_root not in runtime_root.parents
print("Git commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip())


In [ ]:
run_id = "single-gpu-smoke_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
run_dir = runtime_root / "jupyter-validation" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
environment = os.environ.copy()
environment["HF_HUB_OFFLINE"] = "1"
environment["TRANSFORMERS_OFFLINE"] = "1"

for precision in ("fp32", "amp"):
    command = [
        sys.executable, "training/smoke.py",
        "--detector_path", "training/config/detector/dfd_hr.yaml",
        "--test_config_path", "training/config/test_config.yaml",
        "--dataset_json_folder", str(json_root),
        "--data_root", str(data_root),
        "--weights_path", str(checkpoint_path),
        "--output_dir", str(run_dir / f"{precision}-work"),
        "--precision", precision,
    ]
    subprocess.run(command, cwd=repo_root, env=environment, check=True)


In [ ]:
reports = {}
required_gradient_categories = {"adapter", "router", "head", "query"}
for precision in ("fp32", "amp"):
    with (run_dir / f"{precision}.json").open(encoding="utf-8") as handle:
        report = json.load(handle)
    assert report["status"] == "ok"
    assert report["batches"] == 2
    assert report["optimizer_updated"]
    assert report["checkpoint"]["roundtrip"]
    assert report["checkpoint"]["temporary_file_absent"]
    final_gradients = report["gradient_steps"][-1]
    assert required_gradient_categories.issubset(final_gradients)
    assert all(final_gradients[name]["finite"] for name in required_gradient_categories)
    reports[precision] = report
    print(precision, {
        "losses": report["losses"],
        "step_seconds": report["step_seconds"],
        "peak_memory_bytes": report["peak_memory_bytes"],
        "checkpoint_roundtrip": report["checkpoint"]["roundtrip"],
    })

assert reports["amp"]["peak_memory_bytes"]["allocated"] <= reports["fp32"]["peak_memory_bytes"]["allocated"]
print("Run directory role: DFDHR_RUNTIME_ROOT/jupyter-validation/<run-id>")
